# TASK-012 full Berton AUTO continuation from TASK-011 seed

Notebook-first reproducibility entry point. It reviews the TASK-011 damped/equilibrium-like seed, runs AUTO-07p equilibrium continuations from that seed using `W_a0` and `H_a3` controls, then parses diagnostics and generates summary outputs.

In [ ]:
from pathlib import Path
import json
import pandas as pd

REPO_ROOT = Path.cwd()
EPISODE = REPO_ROOT / 'episodes' / '06-full-model-auto-seed-continuation'
TASK011 = REPO_ROOT / 'episodes' / '05-full-model-oscillatory-orbit' / 'outputs' / 'task011'
seed = pd.read_csv(TASK011 / 'continuation_equilibrium_seed.csv')
verdict = json.loads((TASK011 / 'classification_verdict.json').read_text())
seed, verdict

Because TASK-011 classified the trajectory as damped/equilibrium-like, TASK-012 uses equilibrium continuation rather than periodic-orbit continuation. Controls are `W_a0` (updraft amplitude) and `H_a3` (local humidity profile node), not the previously insensitive `z_W0` branch alone.

In [ ]:
# AUTO Python interface is available when interactive AUTO use is needed:
# import sys
# sys.path.append('/usr/local/lib64/auto-07p/python')
# import auto

import subprocess
subprocess.run(['bash', str(EPISODE / 'auto' / 'berton_full_task012' / 'run_auto.sh')], cwd=REPO_ROOT, check=True)

In [ ]:
subprocess.run(['uv', 'run', 'python', str(EPISODE / 'scripts' / 'berton_full_task012_seed_continuation.py')], cwd=REPO_ROOT, check=True)

In [ ]:
OUT = EPISODE / 'outputs' / 'task012'
summary = pd.read_csv(OUT / 'continuation_summary.csv')
probe = pd.read_csv(OUT / 'python_equilibrium_control_probe.csv')
summary

In [ ]:
probe.groupby('control').agg(
    control_min=('control_value', 'min'),
    control_max=('control_value', 'max'),
    n=('control_value', 'size'),
    success=('success', 'sum'),
    rhs_max=('rhs_norm', 'max'),
    critical_real_min=('critical_real_s_inv', 'min'),
    critical_real_max=('critical_real_s_inv', 'max'),
)

AUTO accepted the seed and reported the TASK-011 stable complex pair, but each first continuation step retried down to the minimum step and ended at `MX`. The independent Python root-continuation probe is retained as a sensitivity check: `W_a0` changes the local equilibrium altitude while remaining stable over 0.1--1.2 m/s; `H_a3` changes the local linear stability proxy and crosses positive critical real part near the seed, suggesting a local humidity-control mechanism worth a scaled AUTO follow-up.